# Local Dataverse upload notebook

Aquest notebook esta adaptat per executar-se localment amb Jupyter, no amb Google Colab.

Abans d'executar la cel.la de pujada, comprova que el fitxer Excel i els fitxers a pujar siguin accessibles des d'aquest directori.


# Script for Uploading Files Automatically

If you have doubts about the code, contact rdr-contacte@csuc.cat

## Script Objective
The main objective of this script is to automatically upload files to a dataset with their respective metadata placed in an Excel file.

## Local execution

Run this notebook with Jupyter from the folder that contains the Excel metadata file and the files to upload.

The first four columns of the Excel file are interpreted in this order, regardless of the displayed language of the headers:

- File Name / Nom del fitxer
- Description / Descripcio
- File Path / Ruta del fitxer
- Tag / Etiqueta

For local uploads, `File Path / Ruta del fitxer` can be an absolute or relative local folder containing the file. If it is not a local folder, it is used as the Dataverse `directoryLabel`.


In [ ]:
# Local Dataverse upload configuration and execution
import importlib.util
import os
import subprocess
import sys
from pathlib import Path


def ensure_package(import_name, package_name=None):
    """Install a package only when it is missing in the local environment."""
    if importlib.util.find_spec(import_name) is None:
        package_name = package_name or import_name
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])


ensure_package("pandas")
ensure_package("openpyxl")
ensure_package("pyDataverse")

import pandas as pd
from pyDataverse.api import NativeApi, DataAccessApi
from pyDataverse.models import Datafile

# Metadata provided for this local upload.
identifier = "3230"
token = os.environ.get("DATAVERSE_API_TOKEN", "YOUR_DATAVERSE_API_TOKEN")
excel_file_name = "wtAcetLys.xlsx"
base_url = "https://dataverse.csuc.cat/"

working_dir = Path.cwd()
doi = f"doi:10.34810/data{identifier}"

api = NativeApi(base_url, token)
data_api = DataAccessApi(base_url, token)


def is_blank(value):
    return pd.isna(value) or str(value).strip() == ""


def resolve_upload_path(file_name, file_path_value):
    """Resolve a local file path from the Excel file name and optional path column."""
    candidates = []
    file_name_path = Path(str(file_name)).expanduser()

    if file_name_path.is_absolute():
        candidates.append(file_name_path)
    else:
        candidates.append(working_dir / file_name_path)

    if not is_blank(file_path_value):
        path_value = Path(str(file_path_value)).expanduser()
        if path_value.is_dir():
            candidates.insert(0, path_value / file_name_path.name)
        elif path_value.is_absolute() and path_value.name == file_name_path.name:
            candidates.insert(0, path_value)
        elif not path_value.is_absolute():
            candidates.insert(0, working_dir / path_value / file_name_path.name)

    for candidate in candidates:
        if candidate.is_file():
            return candidate

    return candidates[0]


def dataverse_directory_label(file_path_value):
    """Use only non-local path values as Dataverse directory labels."""
    if is_blank(file_path_value):
        return None

    path_value = Path(str(file_path_value)).expanduser()
    possible_local_path = path_value if path_value.is_absolute() else working_dir / path_value
    if possible_local_path.exists():
        return None

    return str(file_path_value).strip()


def upload_files(base_url, token, doi, excel_file_name):
    """Upload files to a Dataverse dataset using metadata from an Excel file."""
    excel_path = Path(excel_file_name).expanduser()
    if not excel_path.is_absolute():
        excel_path = working_dir / excel_path

    if not excel_path.is_file():
        raise FileNotFoundError(f"Metadata file not found: {excel_path}")

    metadata = pd.read_excel(excel_path)
    if metadata.shape[1] < 4:
        raise ValueError("The Excel file must contain at least four columns: file name, description, file path and tag.")

    rows = metadata.iloc[:, :4].to_dict("records")
    files_to_upload = []
    missing_files = []

    for row in rows:
        values = list(row.values())
        file_name, description, file_path_value, tags = values
        if is_blank(file_name):
            continue

        upload_path = resolve_upload_path(file_name, file_path_value)
        if upload_path.is_file():
            files_to_upload.append((str(file_name).strip(), upload_path, description, file_path_value, tags))
        else:
            missing_files.append(str(upload_path))

    if missing_files:
        print("No files uploaded. These files were not found:")
        for missing_file in missing_files:
            print(f"- {missing_file}")
        return

    dataset_response = api.get_dataset(doi)
    if dataset_response.status_code >= 400:
        raise RuntimeError(f"Incorrect token or DOI not found: {doi}. HTTP status: {dataset_response.status_code}")

    for file_name, upload_path, description, file_path_value, tags in files_to_upload:
        datafile = Datafile()
        datafile.set({"pid": doi})
        datafile.set({"filename": Path(file_name).name})

        if not is_blank(description):
            datafile.set({"description": str(description).strip()})

        directory_label = dataverse_directory_label(file_path_value)
        if directory_label:
            datafile.set({"directoryLabel": directory_label})

        if not is_blank(tags):
            categories = [tag.strip() for tag in str(tags).split(",") if tag.strip()]
            if categories:
                datafile.set({"categories": categories})

        print(f"Starting upload: {file_name}")
        response = api.upload_datafile(doi, str(upload_path), datafile.json())
        if response.status_code >= 400:
            print(f"Upload failed: {file_name}. HTTP status: {response.status_code}")
            print(response.text)
        else:
            print(f"Finished upload: {file_name}")


upload_files(base_url, token, doi, excel_file_name)


In [ ]:
# Run this cell after uploading to get the size of the dataset.
def filemetadata(base_url, token, doi, filemetadata_keys, filemetadata_values):
    from pyDataverse.api import NativeApi

    api = NativeApi(base_url, token)
    dataset_response = api.get_dataset(doi)
    if dataset_response.status_code >= 400:
        raise RuntimeError(f"Could not read dataset metadata for {doi}. HTTP status: {dataset_response.status_code}")

    files = dataset_response.json().get("data", {}).get("latestVersion", {}).get("files", [])
    for file_info in files:
        filemetadata_resp = file_info.get("dataFile", {})
        filemetadata_keys.append(list(filemetadata_resp.keys()))
        filemetadata_values.append(list(filemetadata_resp.values()))


def format_size(size_in_bytes):
    units = ["Bytes", "KB", "MB", "GB", "TB"]
    size = float(size_in_bytes)
    unit_index = 0

    while size >= 1024 and unit_index < len(units) - 1:
        size /= 1024
        unit_index += 1

    return f"{size:.2f} {units[unit_index]}"


def get_index(key_list, key):
    return key_list.index(key) if key in key_list else None


def get_size(entry, key_list):
    original_index = get_index(key_list, "originalFileSize")
    file_index = get_index(key_list, "filesize")
    if original_index is not None and isinstance(entry[original_index], int):
        return entry[original_index]
    if file_index is not None:
        return entry[file_index]
    return 0


filemetadata_keys = []
filemetadata_values = []
filemetadata(base_url, token, doi, filemetadata_keys, filemetadata_values)

if not filemetadata_values:
    print("No files found in the dataset.")
else:
    sizes = [get_size(entry, filemetadata_keys[i]) for i, entry in enumerate(filemetadata_values)]
    archival_sizes = []
    for entry, keys in zip(filemetadata_values, filemetadata_keys):
        filesize_index = get_index(keys, "filesize")
        if filesize_index is not None:
            archival_sizes.append(entry[filesize_index])

    print("Total original format dataset size:", format_size(sum(sizes)))
    print("Total archival format dataset size:", format_size(sum(archival_sizes)))
